# 01 — Feature Engineering
Builds the daily (Reference, Size) demand panel used for forecasting.

**Inputs (place in this same folder):**
- `Customer_Order.csv`
- `Product.csv`

**Output:**
- `forecasting_features.parquet` / `.csv`

Run every cell top to bottom once.

In [ ]:
import pandas as pd, numpy as np, os

DATA_DIR = "."
OUT_DIR  = "."
TEST_HORIZON_DAYS = 20   # business days held out for testing

In [ ]:
import os as _os
# ── Image output folder ─────────────────────────────────────────────────────
# All figures will be saved to a subfolder called "figures" inside the
# directory where you run this notebook.  Change FIGURES_DIR if you prefer
# a different path.
FIGURES_DIR = _os.path.join(_os.getcwd(), "figures")
_os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"[INFO] Figures will be saved to: {FIGURES_DIR}")


## 1. Load and clean raw order data

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/Customer_Order.csv", sep=';', encoding='utf-8-sig')
df.columns = [c.strip() for c in df.columns]
df['Reference'] = df['Reference'].str.strip()
df['creationDate'] = pd.to_datetime(df['creationDate'], format='%d/%m/%Y %H:%M', errors='coerce')
df['date'] = df['creationDate'].dt.normalize()

prod = pd.read_csv(f"{DATA_DIR}/Product.csv", sep=';', encoding='utf-8-sig')
prod.columns = [c.strip() for c in prod.columns]
for c in prod.columns:
    if prod[c].dtype == object:
        prod[c] = prod[c].str.strip()

df.head()

## 2. Restrict to the real operating calendar
The warehouse operates Monday-Friday (weekends carry <2% of order lines and Saturdays never
appear). Keeping only business days turns artificial "closed" zeros into a clean calendar.

In [ ]:
df = df[df['date'].dt.dayofweek < 5].copy()
start, end = df['date'].min(), df['date'].max()
biz_days = pd.bdate_range(start, end)
print(f"Business days: {len(biz_days)}  ({start.date()} -> {end.date()})")

## 3. Define the target
`picks` = number of order lines per day (≈ pick trips) — this is what drives picker travel,
so it's the primary target. `units` is kept as an alternative.

In [ ]:
g = (df.groupby(['Reference', 'Size (US)', 'date'])
       .agg(picks=('orderNumber', 'size'), units=('quantity (units)', 'sum'))
       .reset_index())
g.head()

## 4. Build the dense panel
Every (Reference, Size) combo gets a continuous row for every business day from its **debut**
(first ever pick) onward, zero-filled where nothing happened.

In [ ]:
frames = []
for (ref, size), sub in g.groupby(['Reference', 'Size (US)']):
    idx = biz_days[biz_days >= sub['date'].min()]
    s = sub.set_index('date').reindex(idx).assign(Reference=ref, **{'Size (US)': size})
    s['picks'] = s['picks'].fillna(0).astype(int)
    s['units'] = s['units'].fillna(0)
    frames.append(s.reset_index().rename(columns={'index': 'date'}))

panel = pd.concat(frames, ignore_index=True).sort_values(
    ['Reference', 'Size (US)', 'date']).reset_index(drop=True)
panel = panel.merge(prod, on='Reference', how='left')
print(panel.shape)
panel.head()

## 5. Feature engineering
Every feature below uses information through **t-1 only** — nothing from the day being
predicted ever leaks in.

In [ ]:
grp = panel.groupby(['Reference', 'Size (US)'])['picks']

# lags (business-day steps: 1,2,3 days; 1wk=5; 2wk=10; 4wk=20)
for L in [1, 2, 3, 5, 10, 20]:
    panel[f'lag_{L}'] = grp.shift(L)

# rolling mean / std, shifted so "today" is excluded
for W in [5, 10, 20]:
    panel[f'rmean_{W}'] = grp.transform(lambda s: s.shift(1).rolling(W, min_periods=1).mean())
    panel[f'rstd_{W}']  = grp.transform(lambda s: s.shift(1).rolling(W, min_periods=1).std())

# intermittent-demand signals (Croston-style decomposition)
for W in [10, 20, 60]:
    panel[f'freq_{W}'] = grp.transform(lambda s: (s.shift(1) > 0).rolling(W, min_periods=1).mean())
panel['nz_mean_20'] = grp.transform(lambda s: s.shift(1).where(s.shift(1) > 0).rolling(20, min_periods=1).mean())
panel['expanding_mean'] = grp.transform(lambda s: s.shift(1).expanding().mean())

def days_since(s):
    pos = np.arange(len(s))
    last = np.where(s.values > 0, pos, np.nan)
    last = pd.Series(last).shift(1).ffill().values
    return pos - last

panel['days_since_last'] = panel.groupby(['Reference', 'Size (US)'])['picks'].transform(days_since)
panel['series_age'] = panel.groupby(['Reference', 'Size (US)']).cumcount()

panel['dow']        = panel['date'].dt.dayofweek
panel['weekofyear'] = panel['date'].dt.isocalendar().week.astype(int)
panel['month']      = panel['date'].dt.month
panel['dayofmonth'] = panel['date'].dt.day

lr = [c for c in panel.columns if c.startswith(('lag_', 'rmean_', 'rstd_', 'freq_', 'nz_mean', 'expanding'))]
panel[lr] = panel[lr].fillna(0)
panel['days_since_last'] = panel['days_since_last'].fillna(panel['series_age'])

for c in ['ABCCOD', 'Sector']:
    panel[c] = panel[c].astype('category')
    panel[f'{c}_code'] = panel[c].cat.codes

print("Feature columns added:", lr + ['days_since_last','series_age','dow','weekofyear','month','dayofmonth'])

## 6. Temporal train/test split (last 20 business days = test, never used for training)

In [ ]:
cutoff = biz_days[-TEST_HORIZON_DAYS]
panel['split'] = np.where(panel['date'] < cutoff, 'train', 'test')
print("Cutoff date:", cutoff.date())
print(panel['split'].value_counts())

## 7. Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
try:
    panel.to_parquet(f"{OUT_DIR}/forecasting_features.parquet", index=False)
except Exception as e:
    print("Parquet save skipped (pip install pyarrow to enable):", e)
panel.to_csv(f"{OUT_DIR}/forecasting_features.csv", index=False)
print("Saved. Final shape:", panel.shape)